<a href="https://colab.research.google.com/github/MariaMuu/Thesis/blob/main/wikidata%2Bllm%20query.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install openai

In [11]:
import json
import requests
import os
from openai import OpenAI
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')


client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])


# ─── Step 1: Extract entities + relations + intent via OpenAI ─────────────────

def extract_components(question: str) -> dict:
    prompt = f"""
You are an NLP assistant for Wikidata SPARQL query generation.
Given a natural language question, extract the following as JSON:

- entities: list of named entities (people, places, things)
- relations: list of relations/properties being asked about
- intent: one of SELECT, ASK, COUNT
- filters: any constraints (dates, numbers, etc.)
- answer_type: what kind of value is expected (e.g. place, date, person, number)

Question: "{question}"

Respond ONLY with a valid JSON object, no explanation.
"""
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw_response_content = response.choices[0].message.content
    # Remove markdown code block delimiters if present
    if raw_response_content.startswith('```json') and raw_response_content.endswith('```'):
        raw_response_content = raw_response_content[len('```json'):-len('```')].strip()
    try:
        return json.loads(raw_response_content)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from OpenAI. Raw response: '{raw_response_content}'")
        raise e

In [12]:
# ─── Step 2: Link entities to Wikidata Q/P numbers via Falcon 2.0 ─────────────

def link_to_wikidata(question: str) -> dict:
    url = "https://labs.tib.eu/falcon/falcon2/api?mode=short"
    headers = {"Content-Type": "application/json"}
    payload = {"text": question}
    response = requests.post(url, headers=headers, json=payload)
    data = response.json()

    entities = [
        {"label": e.get("label"), "uri": e.get("uri")}
        for e in data.get("entities_wikidata", [])
    ]
    relations = [
        {"label": r.get("label"), "uri": r.get("uri")}
        for r in data.get("relations_wikidata", [])
    ]
    return {"entities": entities, "relations": relations}

In [13]:
# ─── Step 3: Generate SPARQL query via OpenAI ─────────────────────────────────

def generate_sparql(question: str, components: dict, linked: dict) -> str:
    prompt = f"""
You are a SPARQL expert for Wikidata.
Generate a valid Wikidata SPARQL query for the following question.

Question: "{question}"

Extracted components:
{json.dumps(components, indent=2)}

Wikidata linked entities and relations:
{json.dumps(linked, indent=2)}

Rules:
- Use the Wikidata SPARQL endpoint format (wd:, wdt:, p:, ps:, pq:)
- Use SERVICE wikibase:label for labels
- Use LIMIT 10 unless a specific count is requested
- Return ONLY the SPARQL query, no explanation

SPARQL:
"""
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw_sparql_query = response.choices[0].message.content.strip()
    # Remove markdown code block delimiters if present
    if raw_sparql_query.startswith('```sparql') and raw_sparql_query.endswith('```'):
        raw_sparql_query = raw_sparql_query[len('```sparql'):-len('```')].strip()
    return raw_sparql_query

In [14]:
# ─── Step 4: Run the SPARQL query against Wikidata ────────────────────────────

def run_sparql(query: str) -> list:
    url = "https://query.wikidata.org/sparql"
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "Colab Wikidata Query Agent/1.0 (Python Requests)"
    }
    response = requests.get(url, params={"query": query}, headers=headers)
    print(f"Raw SPARQL response status: {response.status_code}")
    print(f"Raw SPARQL response text: {response.text}")
    response.raise_for_status() # Raise an exception for bad status codes
    data = response.json()
    return data["results"]["bindings"]

In [26]:
# ─── Step 4: Run the SPARQL query against Wikidata ────────────────────────────

def run_sparql(query: str) -> list:
    url = "https://query.wikidata.org/sparql"
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "Colab Wikidata Query Agent/1.0 (Python Requests)"
    }
    response = requests.get(url, params={"query": query}, headers=headers)
    print(f"Raw SPARQL response status: {response.status_code}")
    print(f"Raw SPARQL response text: {response.text}")
    response.raise_for_status() # Raise an exception for bad status codes
    data = response.json()
    return data["results"]["bindings"]


# ─── Main pipeline ────────────────────────────────────────────────────────────

def answer_question(question: str):
    print(f"\nQuestion: {question}\n")

    print("Step 1: Extracting components...")
    components = extract_components(question)
    print(json.dumps(components, indent=2))

    print("\nStep 2: Linking to Wikidata...")
    linked = link_to_wikidata(question)
    print(json.dumps(linked, indent=2))

    print("\nStep 3: Generating SPARQL...")
    sparql = generate_sparql(question, components, linked)
    print(sparql)

    print("\nStep 4: Querying Wikidata...")
    results = run_sparql(sparql)
    print(f"Got {len(results)} result(s):")
    for r in results:
        for key, val in r.items():
            print(f"  {key}: {val['value']}")

    return results


# ─── Run it ───────────────────────────────────────────────────────────────────

#if __name__ == "__main__":
 #   answer_question("What is the capital of the country where the Eiffel Tower is located?")

### Answering the question using the Wikidata pipeline

In [27]:
# Define the question once here
user_question = "What is the capital of the country where the Eiffel Tower is located?"

In [28]:
print(f"\nRunning Wikidata pipeline for question: {user_question}\n")
answer_question(user_question)


Running Wikidata pipeline for question: What is the capital of the country where the Eiffel Tower is located?


Question: What is the capital of the country where the Eiffel Tower is located?

Step 1: Extracting components...
{
  "entities": [
    "Eiffel Tower"
  ],
  "relations": [
    "capital"
  ],
  "intent": "SELECT",
  "filters": [],
  "answer_type": "place"
}

Step 2: Linking to Wikidata...
{
  "entities": [
    {
      "label": null,
      "uri": null
    }
  ],
  "relations": []
}

Step 3: Generating SPARQL...
SELECT ?capitalLabel WHERE {
  wd:Q243 wdt:P17 ?country.
  ?country wdt:P36 ?capital.
  SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }
}
LIMIT 10

Step 4: Querying Wikidata...
Raw SPARQL response status: 200
Raw SPARQL response text: {
  "head" : {
    "vars" : [ "capitalLabel" ]
  },
  "results" : {
    "bindings" : [ {
      "capitalLabel" : {
        "xml:lang" : "en",
        "type" : "literal",
        "value" : "Paris"
      }


[{'capitalLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'Paris'}}]

## Get user's question and ask the LLM

In [29]:
def ask_llm_directly(question: str) -> str:
    prompt = f"""
    Answer the following question in one word.

    Question: "{question}"

    Answer:
    """
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

In [30]:
if __name__ == "__main__":
    question_for_llm = user_question
    print(f"\nQuestion for LLM: {question_for_llm}\n")
    llm_answer = ask_llm_directly(question_for_llm)
    print(f"LLM's Answer: {llm_answer}")


Question for LLM: What is the capital of the country where the Eiffel Tower is located?

LLM's Answer: Paris
